# Baseline — BoW

BoW (Bag-of-Words) vektörleştirme ile temel ML sınıflandırıcılarının baseline performansı.
Diğer deneylerin karşılaştırma referansı olarak kullanılır.

### Kütüphaneler

In [1]:
import re
import pandas as pd
import numpy as np
import time
import copy
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import xgboost as xgb

### Veri Yükleme

In [2]:
df = pd.read_csv('spam_cleaned.csv', encoding='latin1')

In [3]:
X, y = df['sms'], df['type']
print(f"Veri seti: {len(df)} satır | ham: {(y==0).sum()}, spam: {(y==1).sum()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=111, stratify=y
)
print(f"Eğitim: {len(X_train)} | Test: {len(X_test)} | Train spam oranı: {y_train.mean()*100:.1f}%")


Veri seti: 5570 satır | ham: 4823, spam: 747
Eğitim: 3899 | Test: 1671 | Train spam oranı: 13.4%


### BoW Vektörleştirme

Kelime frekanslarına dayalı temsil. Gramer ve sıra göz ardı edilir.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)
print(f"BoW özellik boyutu: {X_train_vec.shape[1]}")

BoW özellik boyutu: 6911


### Değerlendirme Fonksiyonu

In [5]:
import scipy.stats as stats

def evaluate(name, clf, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred = clf.predict(X_te)
    try:
        y_score = clf.predict_proba(X_te)[:, 1]
    except AttributeError:
        y_score = clf.decision_function(X_te)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='accuracy')
    cv_acc = cv_scores.mean()
    cv_std = cv_scores.std()
    ci = stats.t.interval(0.95, df=len(cv_scores)-1, loc=cv_acc, scale=stats.sem(cv_scores))
    return {
        'Model':        name,
        'Accuracy':     round(accuracy_score(y_te, y_pred), 4),
        'Precision':    round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall':       round(recall_score(y_te, y_pred), 4),
        'F1':           round(f1_score(y_te, y_pred), 4),
        'ROC-AUC':      round(roc_auc_score(y_te, y_score), 4),
        'CV-Acc':       round(cv_acc, 4),
        'CV-Std':       round(cv_std, 4),
        'CI-95% Low':   round(ci[0], 4),
        'CI-95% High':  round(ci[1], 4),
        'Time(s)':      round(elapsed, 4),
    }

### Sınıflandırıcılar

LR, DT, KNN, SVM, NC — tüm metrikler raporlanır.

In [6]:
classifiers = [
    ('LR',  LogisticRegression(max_iter=1000)),
    ('DT',  DecisionTreeClassifier(random_state=42)),
    ('KNN', KNeighborsClassifier(n_neighbors=3)),
    ('SVM', SVC(kernel='linear', probability=True)),
    ('NC',  NearestCentroid()),
]

rows = []
for name, clf in classifiers:
    rows.append(evaluate(name, copy.deepcopy(clf), X_train_vec, y_train, X_test_vec, y_test))

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

Model  Accuracy  Precision  Recall     F1  ROC-AUC  CV-Acc  CV-Std  CI-95% Low  CI-95% High  Time(s)
   LR    0.9838     0.9713  0.9062 0.9376   0.9877  0.9820  0.0044      0.9760       0.9881   0.0226
   DT    0.9731     0.9124  0.8839 0.8980   0.9354  0.9690  0.0021      0.9661       0.9718   0.0344
  KNN    0.9659     0.9326  0.8036 0.8633   0.9316  0.9623  0.0029      0.9583       0.9663   0.0002
  SVM    0.9844     0.9670  0.9152 0.9404   0.9831  0.9820  0.0039      0.9766       0.9875   0.7088
   NC    0.9575     0.8964  0.7723 0.8297   0.9955  0.9487  0.0049      0.9419       0.9555   0.0650
